# LeNet-5 Implementation in PyTorch

This notebook contains the implementation of the LeNet-5 architecture using PyTorch.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        # Layer 1: Convolutional layer with 6 filters of size 5x5, padding 2
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=2)
        # Layer 2: Max Pooling layer with 2x2 kernel and stride 2
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Layer 3: Convolutional layer with 16 filters of size 5x5
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        # Layer 4: Max Pooling layer with 2x2 kernel and stride 2
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Layer 5: Convolutional layer with 120 filters of size 5x5
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1)

        # Layer 6: Fully connected layer
        self.fc1 = nn.Linear(in_features=120, out_features=84)

        # Layer 7: Output layer
        self.fc2 = nn.Linear(in_features=84, out_features=num_classes)

    def forward(self, x):
        # Pass through first conv + pool
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # Pass through second conv + pool
        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        # Pass through third conv
        x = F.relu(self.conv3(x))

        # Flatten and pass through fully connected layers
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))

        # Output layer (no activation here, usually handled by CrossEntropyLoss)
        x = self.fc2(x)
        return x

In [2]:
# Instantiate the model
model = LeNet5(num_classes=10)
print(model)

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
)


In [3]:
# Visualização dos filtros convolucionais iniciais (antes do treinamento)
import math
import matplotlib.pyplot as plt

with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            weights = module.weight.detach().cpu()

            # Mostra todos os filtros de saída usando o primeiro canal de entrada
            filters = weights[:, 0, :, :]
            n_filters = filters.shape[0]
            cols = min(10, n_filters)
            rows = math.ceil(n_filters / cols)

            fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.6))
            axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

            for i in range(len(axes)):
                ax = axes[i]
                ax.axis("off")
                if i < n_filters:
                    ax.imshow(filters[i], cmap="gray")
                    ax.set_title(f"f{i}", fontsize=8)

            fig.suptitle(f"{name} | shape={tuple(weights.shape)}", fontsize=12)
            plt.tight_layout()
            plt.show()

conv1: shape=(6, 1, 5, 5)
  mean=0.007293, std=0.120169, min=-0.197918, max=0.199477
  Primeiro filtro (canal 0):
tensor([[-0.0040,  0.1843,  0.1092, -0.0069,  0.1790],
        [ 0.1740, -0.0800,  0.1734, -0.0029,  0.0737],
        [-0.1624,  0.1405,  0.1371,  0.0618,  0.1620],
        [ 0.1871,  0.0796,  0.0387, -0.1097, -0.1102],
        [ 0.0926, -0.0239, -0.1179,  0.1805, -0.0690]])
------------------------------------------------------------
conv2: shape=(16, 6, 5, 5)
  mean=-0.002327, std=0.047025, min=-0.081600, max=0.081528
  Primeiro filtro (canal 0):
tensor([[-0.0209,  0.0236, -0.0762, -0.0312, -0.0481],
        [-0.0750,  0.0484,  0.0432, -0.0169, -0.0246],
        [-0.0337, -0.0530,  0.0201,  0.0570, -0.0530],
        [-0.0253,  0.0807,  0.0248, -0.0153,  0.0289],
        [ 0.0150,  0.0191,  0.0576,  0.0089, -0.0074]])
------------------------------------------------------------
conv3: shape=(120, 16, 5, 5)
  mean=-0.000171, std=0.028906, min=-0.049998, max=0.050000
  Prime